In [36]:
# Importamos librerias
import yfinance as yf
import numpy as np
import pandas as pd
import math


In [46]:
# Lista con los tickers de las acciones
acciones = ['STX', 'NVDA', 'VRTX','NVO']
fecha_inicio = '2020-1-1'
fecha_final = '2026-7-28'
# Usar api de yfinance para descargar el precio de los tickers y extraer solo la columna 'Close'
df_acciones = (yf.download(tickers=acciones, start= fecha_inicio, end= fecha_final, multi_level_index= False))['Close']
df_acciones

[*********************100%***********************]  4 of 4 completed


Ticker,NVDA,NVO,STX,VRTX
Date,,,,
2020-01-02,5.963803,25.287043,48.635624,219.449997
2020-01-03,5.868349,24.745424,47.357441,217.979996
2020-01-06,5.892958,24.702095,46.794708,224.029999
2020-01-07,5.964300,24.676094,47.357441,223.789993
2020-01-08,5.975486,24.632769,47.727222,231.089996
...,...,...,...,...
2026-07-21,207.289993,49.380001,891.830017,482.000000
2026-07-22,212.059998,48.189999,908.099976,472.570007
2026-07-23,208.759995,48.180000,913.359985,473.089996


In [47]:
# Calcular rendimiento diario de las acciones
df_rendimiento = ((df_acciones/df_acciones.shift(1))-1).dropna()
df_rendimiento

Ticker,NVDA,NVO,STX,VRTX
Date,,,,
2020-01-03,-0.016006,-0.021419,-0.026281,-0.006699
2020-01-06,0.004194,-0.001751,-0.011883,0.027755
2020-01-07,0.012106,-0.001053,0.012026,-0.001071
2020-01-08,0.001876,-0.001756,0.007808,0.032620
2020-01-09,0.010982,0.010378,0.018360,-0.003592
...,...,...,...,...
2026-07-21,0.019726,-0.004636,0.111384,0.003122
2026-07-22,0.023011,-0.024099,0.018243,-0.019564
2026-07-23,-0.015562,-0.000207,0.005792,0.001100


In [48]:
# Tomar el retorno esperado como el promedio historico anual de rendimiento
re = df_rendimiento.mean()*252
re

Ticker
NVDA    0.669637
NVO     0.170412
STX     0.536530
VRTX    0.171257
dtype: float64

In [49]:
# Calcular la volatilidad de cada activo usando el metodo .std() y ddof = 1 para que sea tomada como sample
volatilidad = (df_rendimiento).std(ddof = 1)*(252**(1/2))
volatilidad

Ticker
NVDA    0.520842
NVO     0.363422
STX     0.458794
VRTX    0.318945
dtype: float64

In [50]:
# Matriz de covarianza
covarianza = df_rendimiento.cov()*252
covarianza

Ticker,NVDA,NVO,STX,VRTX
Ticker,,,,
NVDA,0.271277,0.048401,0.098352,0.042227
NVO,0.048401,0.132075,0.025853,0.032406
STX,0.098352,0.025853,0.210492,0.027430
VRTX,0.042227,0.032406,0.027430,0.101726


In [74]:
# Lista vacia donde se asignaran los pesos por activo
weights = []
# Ciclo while donde si la suma de los pesos no es muy cercana a 1 los volvera a pedir (not invierte el True de la condicion)
while not math.isclose(sum(weights), 1):
# Por cada accion (t) en df_acciones, se solicitara el peso dado por accion y se agregara a la lista weights
    for t in df_acciones:
        weights.append(float(input(f'Selecciona el peso en decimales de {t}:')))
# En caso de que la suma de weights no sea igual a 1, se eliminaran los elementos en la lista para volver a pasar los pesos
    if sum(weights)!= 1:
        print('Error la suma de los pesos no es igual a 1')
        weights.clear()
        

Selecciona el peso en decimales de NVDA: .3
Selecciona el peso en decimales de NVO: .1
Selecciona el peso en decimales de STX: .2
Selecciona el peso en decimales de VRTX: .4


In [75]:
# Visualizar lista con los pesos
for t,w in zip(df_acciones, weights):
    print(f'{t}: {w}')

NVDA: 0.3
NVO: 0.1
STX: 0.2
VRTX: 0.4


In [79]:
# Transformar a arrays la lista de pesos y re para evitar errores de indexacion
w_array = np.array(weights)
re_array = np.array(re)
# Retorno esperado del portafolio
re_portafolio = sum(w_array*re_array)
re_portafolio

np.float64(0.39374128022676474)

In [92]:
print(w_array, re_array, acciones)

[0.3 0.1 0.2 0.4] [0.6696374  0.17041246 0.53652956 0.17125725] ['STX', 'NVDA', 'VRTX', 'NVO']


$$\text{Riesgo del portafolio }= \sqrt{(w^T*\sum{w})} $$

- $\text{w : vector de pesos del portafolio.}$
- $w^T: \text{transpuesta del vector de pesos.}$
- $\sum: \text{matriz de covarianzas de los rendimientos de los activos.}$

In [53]:
# Calcular riesgo del portafolio con multiplicacion matricial (@)
riesgo_portafolio = (w_array.T @ covarianza @ w_array)**(1/2)
riesgo_portafolio

np.float64(0.2811702307903937)

$$\text{Sharpe Ratio} = \frac{E(rp) - rf}{\sigma p}$$

In [70]:
# Obtener tasa libre de riesgo de los bonos del tesoro de USA a 5y a la fecha dada al inicio
rf = yf.download(tickers='^FVX', start=fecha_inicio, end=fecha_final)['Close']
rf = float(rf.iloc[-1])/100
rf

[*********************100%***********************]  1 of 1 completed
C:\Users\jcbro\AppData\Local\Temp\ipykernel_4896\3101202015.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  rf = float(rf.iloc[-1])/100


0.04396999835968018

In [82]:
# Funcion Ratio de sharpe
def sharpe_ratio(retorno_esperado, tasa_libre_riesgo, riesgo_del_portafolio):
    '''
    Esta funcion devuelve el sharpe ratio del portafolio
    '''
    return (retorno_esperado-tasa_libre_riesgo)/riesgo_del_portafolio


In [85]:
# Utilizar funcion de sharpe Ratio
sharpe_ratio_portafolio = sharpe_ratio(re_portafolio, rf, riesgo_portafolio)
sharpe_ratio_portafolio

np.float64(1.2439840479692583)

In [97]:
### RESUMEN DE RESULTADOS
print(f'''
acciones = {list(re.index)}
pesos = {weights}
retorno del portafolio (re) = {round(re_portafolio, 4)}
riesgo del portafolio = {round(riesgo_portafolio, 4)}
Sharpe ratio = {round(sharpe_ratio_portafolio, 4)}
''')


acciones = ['NVDA', 'NVO', 'STX', 'VRTX']
pesos = [0.3, 0.1, 0.2, 0.4]
retorno del portafolio (re) = 0.3937
riesgo del portafolio = 0.2812
Sharpe ratio = 1.244

